In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sentence_transformers import SentenceTransformer


c:\Users\ps302\anaconda3\envs\genai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [3]:
scholar_df = pd.DataFrame({
    "scholar_id": [1, 2, 3, 4],

    "research_interests": [
        "deep learning for flood prediction and hydrological modelling",
        "computer vision and medical image analysis",
        "structural health monitoring using machine learning",
        "remote sensing and climate change"
    ],

    "expertise": [
        "deep learning, hydrology, remote sensing",
        "computer vision, deep learning, medical imaging",
        "machine learning, structural engineering, sensors",
        "remote sensing, GIS, climate modelling"
    ],

    "department": [
        "Civil Engineering",
        "Computer Science",
        "Civil Engineering",
        "Civil Engineering"
    ],

    "university": [
        "IIT Kharagpur",
        "IIT Delhi",
        "IIT Bombay",
        "IIT Kharagpur"
    ],

    "country": [
        "India",
        "India",
        "India",
        "India"
    ],

    "publication_count": [
        12, 25, 18, 30
    ],

    "citation_count": [
        120, 450, 200, 600
    ],

    "years_experience": [
        2, 5, 3, 7
    ]
})

scholar_df

,scholar_id,research_interests,expertise,department,university,country,publication_count,citation_count,years_experience
0,1,deep learning for flood prediction and hydrolo...,"deep learning, hydrology, remote sensing",Civil Engineering,IIT Kharagpur,India,12,120,2
1,2,computer vision and medical image analysis,"computer vision, deep learning, medical imaging",Computer Science,IIT Delhi,India,25,450,5
2,3,structural health monitoring using machine lea...,"machine learning, structural engineering, sensors",Civil Engineering,IIT Bombay,India,18,200,3
3,4,remote sensing and climate change,"remote sensing, GIS, climate modelling",Civil Engineering,IIT Kharagpur,India,30,600,7


In [4]:
scholar_df["research_text"] = (
    "Research interests: "
    + scholar_df["research_interests"].fillna("")
    + ". Expertise: "
    + scholar_df["expertise"].fillna("")
)

In [5]:
for i in scholar_df["research_text"]:
    print(i)

Research interests: deep learning for flood prediction and hydrological modelling. Expertise: deep learning, hydrology, remote sensing
Research interests: computer vision and medical image analysis. Expertise: computer vision, deep learning, medical imaging
Research interests: structural health monitoring using machine learning. Expertise: machine learning, structural engineering, sensors
Research interests: remote sensing and climate change. Expertise: remote sensing, GIS, climate modelling


In [5]:
text_encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

text_encoder.to(device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7654.25it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [6]:
print(
    text_encoder.get_embedding_dimension()
)

384


In [8]:
texts = scholar_df["research_text"].tolist()

text_embeddings = text_encoder.encode(
    texts,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

text_embeddings = text_embeddings.to(device)

print(text_embeddings.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00, 26.60it/s]

torch.Size([4, 384])


In [9]:
# text_embeddings[0]
text_01 = "Applying deep learning to satellite imagery for advanced flood forecasting. This research integrates neural networks with remote sensing data to model complex hydrological systems"
text_01

'Applying deep learning to satellite imagery for advanced flood forecasting. This research integrates neural networks with remote sensing data to model complex hydrological systems'

In [13]:
import torch

# Slice the two vectors
embedding1 = text_embeddings[0]
embedding2 = text_encoder.encode(
    text_01,
    normalize_embeddings = True,
    convert_to_tensor = True
)

# Calculate similarity (dot product)
similarity = torch.dot(embedding1, embedding2).item()

print(f"Similarity score: {similarity:.4f}")


Similarity score: 0.8121


In [15]:
# Select your target tensor (e.g., the first paper)
target_embedding = embedding2

# Multiply target by the entire matrix (matrix-vector multiplication)
# unsqueeze adds a dimension to make it a 2D matrix for calculation
similarities = torch.matmul(text_embeddings, target_embedding.unsqueeze(1)).squeeze()

# Get the top 5 most similar matches (excluding itself at index 0)
top_k = torch.topk(similarities, k=4)

for score, idx in zip(top_k.values[0:], top_k.indices[0:]):
    print(f"Index: {idx.item()} | Similarity: {score.item():.4f}")


Index: 0 | Similarity: 0.8121
Index: 3 | Similarity: 0.4374
Index: 1 | Similarity: 0.3443
Index: 2 | Similarity: 0.2779
